# Week 4: Data Analysis - Solutions

Solutions for NumPy, Pandas, and visualization problems.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

## Part 1: NumPy for Sequences

### Problem 1.1: One-Hot Encoding

In [ ]:
def one_hot_encode(sequence):
    """One-hot encode a DNA sequence.
    
    Returns array of shape (len(sequence), 4) for A, C, G, T.
    """
    mapping = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    seq_len = len(sequence)
    encoding = np.zeros((seq_len, 4), dtype=np.float32)
    
    for i, nuc in enumerate(sequence.upper()):
        if nuc in mapping:
            encoding[i, mapping[nuc]] = 1.0
    
    return encoding

def decode_one_hot(encoding):
    """Decode one-hot back to sequence."""
    nucleotides = 'ACGT'
    indices = np.argmax(encoding, axis=1)
    return ''.join(nucleotides[i] for i in indices)

# Test
seq = "ATGCGATCGA"
encoded = one_hot_encode(seq)
print(f"Original: {seq}")
print(f"Encoded shape: {encoded.shape}")
print(f"First 3 positions:\n{encoded[:3]}")
print(f"Decoded: {decode_one_hot(encoded)}")

### Problem 1.2: Position Weight Matrix

In [ ]:
def build_pwm(sequences, pseudocount=0.5):
    """Build a Position Weight Matrix from aligned sequences.
    
    Returns PWM with log-odds scores.
    """
    if not sequences:
        return None
    
    seq_len = len(sequences[0])
    n_seqs = len(sequences)
    
    # Count matrix
    counts = np.zeros((4, seq_len))
    nuc_idx = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    
    for seq in sequences:
        for i, nuc in enumerate(seq.upper()):
            if nuc in nuc_idx:
                counts[nuc_idx[nuc], i] += 1
    
    # Add pseudocounts and convert to frequencies
    counts += pseudocount
    freqs = counts / counts.sum(axis=0)
    
    # Convert to log-odds (assuming background = 0.25)
    pwm = np.log2(freqs / 0.25)
    
    return pwm

def score_sequence(sequence, pwm):
    """Score a sequence against a PWM."""
    nuc_idx = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    score = 0.0
    for i, nuc in enumerate(sequence.upper()):
        if nuc in nuc_idx:
            score += pwm[nuc_idx[nuc], i]
    return score

def scan_sequence(sequence, pwm, threshold=0):
    """Scan sequence for PWM matches above threshold."""
    motif_len = pwm.shape[1]
    matches = []
    
    for i in range(len(sequence) - motif_len + 1):
        subseq = sequence[i:i+motif_len]
        score = score_sequence(subseq, pwm)
        if score >= threshold:
            matches.append((i, subseq, score))
    
    return matches

# Test
# Create motif from known binding sites
binding_sites = [
    "GATAAG",
    "GATAAT",
    "GATAAC",
    "GATACG",
    "GATACG"
]

pwm = build_pwm(binding_sites)
print("PWM (log-odds):")
print(pd.DataFrame(pwm, index=['A', 'C', 'G', 'T']))

# Scan for matches
test_seq = "ATGGATAAGGCCGATAATCCC"
matches = scan_sequence(test_seq, pwm, threshold=5)
print(f"\nMatches in '{test_seq}':")
for pos, motif, score in matches:
    print(f"  Position {pos}: {motif} (score: {score:.2f})")

### Problem 1.3: Sliding Window Analysis

In [ ]:
def sliding_window_gc(sequence, window_size, step=1):
    """Calculate GC content in sliding windows using NumPy."""
    # Convert sequence to numeric (G/C = 1, A/T = 0)
    gc_binary = np.array([1 if n in 'GC' else 0 for n in sequence.upper()])
    
    # Number of windows
    n_windows = (len(sequence) - window_size) // step + 1
    
    # Calculate GC for each window
    positions = np.arange(0, n_windows * step, step)
    gc_values = np.array([
        gc_binary[i:i+window_size].mean() * 100 
        for i in positions
    ])
    
    return positions, gc_values

# Optimized version using stride tricks
def sliding_window_gc_fast(sequence, window_size, step=1):
    """Fast sliding window using NumPy stride tricks."""
    gc_binary = np.array([1 if n in 'GC' else 0 for n in sequence.upper()])
    
    # Use cumsum for O(n) calculation
    cumsum = np.concatenate([[0], np.cumsum(gc_binary)])
    
    # Positions for windows
    starts = np.arange(0, len(sequence) - window_size + 1, step)
    ends = starts + window_size
    
    gc_values = (cumsum[ends] - cumsum[starts]) / window_size * 100
    
    return starts, gc_values

# Test
np.random.seed(42)
random_seq = ''.join(np.random.choice(['A', 'T', 'G', 'C'], size=1000))

positions, gc_values = sliding_window_gc_fast(random_seq, window_size=50, step=10)

plt.figure(figsize=(12, 4))
plt.plot(positions, gc_values, 'b-', linewidth=1)
plt.axhline(y=50, color='r', linestyle='--', label='50% GC')
plt.xlabel('Position')
plt.ylabel('GC Content (%)')
plt.title('GC Content Sliding Window Analysis')
plt.legend()
plt.tight_layout()
plt.show()

## Part 2: Pandas for Bioinformatics

### Problem 2.1: Gene Expression Analysis

In [ ]:
# Generate sample gene expression data
np.random.seed(42)
n_genes = 100
n_samples = 6

gene_names = [f"Gene_{i:03d}" for i in range(n_genes)]
sample_names = ['Control_1', 'Control_2', 'Control_3', 'Treatment_1', 'Treatment_2', 'Treatment_3']

# Base expression
base_expr = np.random.lognormal(mean=5, sigma=1.5, size=n_genes)

# Create expression matrix
expression_data = []
for i, sample in enumerate(sample_names):
    noise = np.random.normal(0, 0.5, n_genes)
    fold_change = np.zeros(n_genes)
    
    # Add differential expression for some genes in treatment
    if 'Treatment' in sample:
        # 20 upregulated, 20 downregulated
        fold_change[:20] = np.random.uniform(1, 3, 20)
        fold_change[20:40] = np.random.uniform(-3, -1, 20)
    
    expr = base_expr * (2 ** (fold_change + noise))
    expression_data.append(expr)

# Create DataFrame
df_expr = pd.DataFrame(
    np.array(expression_data).T,
    index=gene_names,
    columns=sample_names
)

print("Expression data shape:", df_expr.shape)
df_expr.head(10)

In [ ]:
def differential_expression_analysis(df, control_cols, treatment_cols):
    """Perform simple differential expression analysis."""
    results = pd.DataFrame(index=df.index)
    
    # Calculate means
    results['control_mean'] = df[control_cols].mean(axis=1)
    results['treatment_mean'] = df[treatment_cols].mean(axis=1)
    
    # Calculate fold change (log2)
    results['log2_fold_change'] = np.log2(
        results['treatment_mean'] / results['control_mean']
    )
    
    # Simple t-test (for demonstration)
    from scipy import stats
    
    p_values = []
    for gene in df.index:
        control_vals = df.loc[gene, control_cols].values
        treatment_vals = df.loc[gene, treatment_cols].values
        _, p_val = stats.ttest_ind(control_vals, treatment_vals)
        p_values.append(p_val)
    
    results['p_value'] = p_values
    
    # FDR correction (Benjamini-Hochberg)
    from scipy.stats import rankdata
    n = len(p_values)
    ranked_p = rankdata(p_values)
    fdr = np.array(p_values) * n / ranked_p
    fdr = np.minimum(fdr, 1.0)  # Cap at 1
    results['fdr'] = fdr
    
    # Significance classification
    results['significant'] = (results['fdr'] < 0.05) & (np.abs(results['log2_fold_change']) > 1)
    results['direction'] = np.where(
        results['log2_fold_change'] > 0, 'up', 'down'
    )
    
    return results.sort_values('p_value')

# Run analysis
control_cols = ['Control_1', 'Control_2', 'Control_3']
treatment_cols = ['Treatment_1', 'Treatment_2', 'Treatment_3']

de_results = differential_expression_analysis(df_expr, control_cols, treatment_cols)
print(f"Significant genes: {de_results['significant'].sum()}")
print(f"  - Upregulated: {((de_results['significant']) & (de_results['direction'] == 'up')).sum()}")
print(f"  - Downregulated: {((de_results['significant']) & (de_results['direction'] == 'down')).sum()}")

de_results.head(10)

### Problem 2.2: Volcano Plot

In [ ]:
def volcano_plot(de_results, fc_threshold=1, p_threshold=0.05):
    """Create a volcano plot for differential expression results."""
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Calculate -log10(p-value)
    neg_log_p = -np.log10(de_results['p_value'].clip(lower=1e-300))
    
    # Color coding
    colors = []
    for i, row in de_results.iterrows():
        if row['significant'] and row['direction'] == 'up':
            colors.append('red')
        elif row['significant'] and row['direction'] == 'down':
            colors.append('blue')
        else:
            colors.append('gray')
    
    # Scatter plot
    ax.scatter(
        de_results['log2_fold_change'],
        neg_log_p,
        c=colors,
        alpha=0.6,
        s=50
    )
    
    # Threshold lines
    ax.axhline(-np.log10(p_threshold), color='gray', linestyle='--', linewidth=1)
    ax.axvline(-fc_threshold, color='gray', linestyle='--', linewidth=1)
    ax.axvline(fc_threshold, color='gray', linestyle='--', linewidth=1)
    
    # Labels for top genes
    top_genes = de_results[de_results['significant']].head(5)
    for gene_name in top_genes.index:
        x = de_results.loc[gene_name, 'log2_fold_change']
        y = -np.log10(de_results.loc[gene_name, 'p_value'])
        ax.annotate(gene_name, (x, y), fontsize=8, 
                   xytext=(5, 5), textcoords='offset points')
    
    ax.set_xlabel('Log2 Fold Change', fontsize=12)
    ax.set_ylabel('-Log10(p-value)', fontsize=12)
    ax.set_title('Volcano Plot: Treatment vs Control', fontsize=14)
    
    # Legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='red', alpha=0.6, label='Upregulated'),
        Patch(facecolor='blue', alpha=0.6, label='Downregulated'),
        Patch(facecolor='gray', alpha=0.6, label='Not significant')
    ]
    ax.legend(handles=legend_elements, loc='upper right')
    
    plt.tight_layout()
    return fig

fig = volcano_plot(de_results)
plt.show()

### Problem 2.3: Expression Heatmap

In [ ]:
def expression_heatmap(df, gene_list=None, z_score=True):
    """Create a heatmap of gene expression."""
    # Select genes
    if gene_list is not None:
        data = df.loc[gene_list]
    else:
        data = df
    
    # Z-score normalization
    if z_score:
        data = data.apply(lambda x: (x - x.mean()) / x.std(), axis=1)
    
    # Create heatmap
    fig, ax = plt.subplots(figsize=(10, max(8, len(data) * 0.3)))
    
    sns.heatmap(
        data,
        cmap='RdBu_r',
        center=0,
        xticklabels=True,
        yticklabels=True,
        ax=ax,
        cbar_kws={'label': 'Z-score' if z_score else 'Expression'}
    )
    
    ax.set_title('Gene Expression Heatmap', fontsize=14)
    ax.set_xlabel('Samples', fontsize=12)
    ax.set_ylabel('Genes', fontsize=12)
    
    plt.tight_layout()
    return fig

# Plot top significant genes
sig_genes = de_results[de_results['significant']].index[:20].tolist()
fig = expression_heatmap(df_expr, gene_list=sig_genes)
plt.show()

## Part 3: Sequence Visualization

### Problem 3.1: GC Landscape Plot

In [ ]:
def gc_landscape_plot(sequence, window_sizes=[50, 100, 200]):
    """Create a multi-scale GC landscape plot."""
    fig, axes = plt.subplots(len(window_sizes), 1, figsize=(14, 3*len(window_sizes)),
                            sharex=True)
    
    if len(window_sizes) == 1:
        axes = [axes]
    
    colors = plt.cm.viridis(np.linspace(0, 0.8, len(window_sizes)))
    
    for ax, window_size, color in zip(axes, window_sizes, colors):
        positions, gc_values = sliding_window_gc_fast(sequence, window_size, step=window_size//5)
        
        # Plot line
        ax.plot(positions, gc_values, color=color, linewidth=1.5, label=f'Window: {window_size}')
        
        # Fill areas
        ax.fill_between(positions, gc_values, 50, 
                       where=(gc_values >= 50), alpha=0.3, color='blue')
        ax.fill_between(positions, gc_values, 50, 
                       where=(gc_values < 50), alpha=0.3, color='red')
        
        ax.axhline(y=50, color='black', linestyle='--', linewidth=0.5)
        ax.set_ylabel('GC %', fontsize=10)
        ax.set_ylim(0, 100)
        ax.legend(loc='upper right')
    
    axes[-1].set_xlabel('Position (bp)', fontsize=12)
    fig.suptitle('GC Content Landscape (Multi-scale)', fontsize=14)
    plt.tight_layout()
    return fig

# Test with random sequence
np.random.seed(42)
# Create sequence with variable GC regions
seq_parts = []
for _ in range(10):
    gc_prob = np.random.uniform(0.3, 0.7)
    weights = [0.5-gc_prob/2, gc_prob/2, gc_prob/2, 0.5-gc_prob/2]
    seq_parts.append(''.join(np.random.choice(['A', 'C', 'G', 'T'], 
                                              size=500, p=weights)))
test_sequence = ''.join(seq_parts)

fig = gc_landscape_plot(test_sequence, window_sizes=[50, 100, 200])
plt.show()

### Problem 3.2: Codon Usage Barplot

In [ ]:
def codon_usage_plot(sequence):
    """Create a codon usage frequency barplot."""
    from collections import Counter
    
    # Extract codons
    codons = [sequence[i:i+3] for i in range(0, len(sequence)-2, 3)]
    codon_counts = Counter(codons)
    
    # Standard codon table (amino acid mapping)
    codon_table = {
        'TTT': 'F', 'TTC': 'F', 'TTA': 'L', 'TTG': 'L',
        'TCT': 'S', 'TCC': 'S', 'TCA': 'S', 'TCG': 'S',
        'TAT': 'Y', 'TAC': 'Y', 'TAA': '*', 'TAG': '*',
        'TGT': 'C', 'TGC': 'C', 'TGA': '*', 'TGG': 'W',
        'CTT': 'L', 'CTC': 'L', 'CTA': 'L', 'CTG': 'L',
        'CCT': 'P', 'CCC': 'P', 'CCA': 'P', 'CCG': 'P',
        'CAT': 'H', 'CAC': 'H', 'CAA': 'Q', 'CAG': 'Q',
        'CGT': 'R', 'CGC': 'R', 'CGA': 'R', 'CGG': 'R',
        'ATT': 'I', 'ATC': 'I', 'ATA': 'I', 'ATG': 'M',
        'ACT': 'T', 'ACC': 'T', 'ACA': 'T', 'ACG': 'T',
        'AAT': 'N', 'AAC': 'N', 'AAA': 'K', 'AAG': 'K',
        'AGT': 'S', 'AGC': 'S', 'AGA': 'R', 'AGG': 'R',
        'GTT': 'V', 'GTC': 'V', 'GTA': 'V', 'GTG': 'V',
        'GCT': 'A', 'GCC': 'A', 'GCA': 'A', 'GCG': 'A',
        'GAT': 'D', 'GAC': 'D', 'GAA': 'E', 'GAG': 'E',
        'GGT': 'G', 'GGC': 'G', 'GGA': 'G', 'GGG': 'G'
    }
    
    # Create DataFrame
    df = pd.DataFrame([
        {'codon': codon, 'count': codon_counts.get(codon, 0), 
         'amino_acid': codon_table.get(codon, 'X')}
        for codon in codon_table.keys()
    ])
    df['frequency'] = df['count'] / df['count'].sum() * 100
    df = df.sort_values(['amino_acid', 'codon'])
    
    # Plot
    fig, ax = plt.subplots(figsize=(16, 6))
    
    # Color by amino acid
    amino_acids = df['amino_acid'].unique()
    colors = plt.cm.tab20(np.linspace(0, 1, len(amino_acids)))
    aa_colors = dict(zip(amino_acids, colors))
    
    bar_colors = [aa_colors[aa] for aa in df['amino_acid']]
    
    bars = ax.bar(range(len(df)), df['frequency'], color=bar_colors)
    ax.set_xticks(range(len(df)))
    ax.set_xticklabels(df['codon'], rotation=90, fontsize=8)
    ax.set_xlabel('Codon', fontsize=12)
    ax.set_ylabel('Frequency (%)', fontsize=12)
    ax.set_title('Codon Usage Frequency', fontsize=14)
    
    plt.tight_layout()
    return fig, df

# Test
np.random.seed(42)
coding_seq = ''.join(np.random.choice(['A', 'T', 'G', 'C'], size=3000))
fig, codon_df = codon_usage_plot(coding_seq)
plt.show()

print("\nTop 10 most frequent codons:")
print(codon_df.nlargest(10, 'count')[['codon', 'amino_acid', 'count', 'frequency']])

### Problem 3.3: K-mer Frequency Distribution

In [ ]:
def kmer_distribution_plot(sequence, k=4):
    """Plot k-mer frequency distribution."""
    from collections import Counter
    
    # Count k-mers
    kmers = [sequence[i:i+k] for i in range(len(sequence) - k + 1)]
    kmer_counts = Counter(kmers)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Top k-mers barplot
    top_kmers = kmer_counts.most_common(20)
    kmers_list, counts_list = zip(*top_kmers)
    
    axes[0].barh(range(len(kmers_list)), counts_list, color='steelblue')
    axes[0].set_yticks(range(len(kmers_list)))
    axes[0].set_yticklabels(kmers_list)
    axes[0].invert_yaxis()
    axes[0].set_xlabel('Count')
    axes[0].set_title(f'Top 20 {k}-mers')
    
    # Frequency distribution histogram
    freq_values = list(kmer_counts.values())
    axes[1].hist(freq_values, bins=50, color='steelblue', edgecolor='white')
    axes[1].set_xlabel('Frequency')
    axes[1].set_ylabel('Number of k-mers')
    axes[1].set_title(f'{k}-mer Frequency Distribution')
    axes[1].axvline(np.mean(freq_values), color='red', linestyle='--', 
                   label=f'Mean: {np.mean(freq_values):.1f}')
    axes[1].legend()
    
    plt.tight_layout()
    
    # Summary stats
    print(f"Total unique {k}-mers: {len(kmer_counts)}")
    print(f"Possible {k}-mers: {4**k}")
    print(f"Coverage: {len(kmer_counts) / (4**k) * 100:.1f}%")
    
    return fig

fig = kmer_distribution_plot(test_sequence, k=4)
plt.show()

---

## Summary

Key techniques demonstrated:

1. **NumPy**: One-hot encoding, PWMs, vectorized sliding windows
2. **Pandas**: Expression data analysis, differential expression, data manipulation
3. **Visualization**: Volcano plots, heatmaps, GC landscapes, codon usage
4. **Statistical Analysis**: t-tests, FDR correction, z-scores